# Integrations — dbt, dlt, and More

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/06_integrations.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/06_integrations.ipynb)

Reuse your dbt schema definitions and plug into 100+ dlt sources — with contract-driven quality gates.

In [ ]:
import subprocess
import sys
import importlib
import urllib.request
import os

if importlib.util.find_spec("lakelogic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lakelogic[polars]"])
if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

---
## 1. dbt Adapter — Reuse Your Schema Definitions

**The Problem:** You already have 200 models defined in dbt `schema.yml`. Rewriting them as LakeLogic contracts doubles your maintenance burden.

**The Solution:** `DataProcessor.from_dbt()` reads your dbt schema and creates a LakeLogic contract from it. Zero duplication.

In [ ]:
from pathlib import Path

# Write a realistic dbt schema.yml
dbt_schema = """
version: 2
models:
  - name: customers
    description: Customer master table
    columns:
      - name: customer_id
        description: Primary key
        tests:
          - not_null
          - unique
      - name: email
        description: Customer email address
        tests:
          - not_null
      - name: first_name
        description: First name
      - name: last_name
        description: Last name
      - name: country
        description: ISO country code
        tests:
          - accepted_values:
              values: ['US', 'GB', 'DE', 'FR', 'JP']
      - name: created_at
        description: Account creation timestamp
        tests:
          - not_null
"""
Path("dbt_schema.yml").write_text(dbt_schema)

# Create a LakeLogic processor from dbt definitions
proc = ll.DataProcessor.from_dbt("dbt_schema.yml", model="customers")
print("Contract created from dbt schema:")
print(f"  Dataset: {proc.contract.dataset}")
print(f"  Fields:  {[f.name for f in proc.contract.model.fields]}")
print(f"  Rules:   {len(proc.contract.quality.row_rules)} row rules")

In [ ]:
# The Proof — generate data and run through the dbt-derived contract
gen = ll.DataGenerator.from_dbt("dbt_schema.yml", model="customers")
source_df = gen.generate(rows=500, invalid_ratio=0.08)

good, bad = proc.run(source_df)
s.assert_reconciliation(source_df, good, bad)
print("\ndbt not_null + accepted_values tests → LakeLogic quality rules. Zero rewrite.")

---
## 2. dlt Adapter — Contract-Driven API Ingestion

**The Problem:** You ingest from GitHub, Stripe, Shopify and 100+ APIs via [dlt](https://dlthub.com). Data arrives with no schema enforcement — bad records flow straight into your warehouse.

**The Solution:** Declare the API directly in your contract's `source.type: dlt` block. LakeLogic extracts the data via dlt's REST API engine, then validates every row through your model and quality rules — all in one `proc.run_source()` call.

In [ ]:
# Install dlt (if not already installed)
import subprocess
import sys

try:
    import dlt
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "dlt"])
    import dlt
print(f"dlt v{dlt.__version__} ready")

In [ ]:
# ── The contract declares the API source directly ─────────────
# No separate dlt script needed — the contract IS the config.
github_contract = s.write_contract(
    """
version: 1.0.0
dataset: github_issues
info:
  title: bronze_github_issues
  domain: engineering
  target_layer: bronze

source:
  type: dlt
  dlt:
    base_url: https://api.github.com
    credentials: {}
    endpoints:
      - name: issues
        path: repos/dlt-hub/dlt/issues
        params:
          state: open
          per_page: 30

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: repository_url
      type: string
    - name: number
      type: integer
      required: true
    - name: title
      type: string
      required: true
    - name: body
      type: string
    - name: state
      type: string
    - name: url
      type: string
    - name: created_at
      type: string
    - name: updated_at
      type: string

quality:
  row_rules:
    - name: valid_state
      sql: "state IN ('open', 'closed')"
    - name: has_title
      sql: "title IS NOT NULL AND title != ''"

server:
  type: local
  path: "."
  schema_policy:
    evolution: "allow"
    unknown_fields: "drop"

""",
    "06_integrations_demo/github_issues.yaml",
)

print("Contract written with source.type=dlt")
print("  API: https://api.github.com/repos/dlt-hub/dlt/issues")
print("  Fields: id, repository_url, number, title, body, state, ...")
print("  Rules: valid_state, has_title")

In [ ]:
# ── One call: dlt extraction + LakeLogic validation ────────────
# run_source() detects source.type=dlt and:
#   1. Builds a dlt REST API pipeline from the contract config
#   2. Extracts data from the GitHub API
#   3. Converts to Polars DataFrame
#   4. Runs schema validation + quality rules
#   5. Returns good/bad split with reconciliation guarantee

proc = ll.DataProcessor(github_contract, engine="polars")
good, bad = proc.run_source()
r = proc.last_report

counts = r.get("counts", {})
print("Contract-driven dlt results:")
print(f"  Source  : {counts.get('source', '?')} issues from GitHub API")
print(f"  Good    : {counts.get('good', '?')} (passed all rules)")
print(f"  Bad     : {counts.get('quarantined', '?')} (quarantined)")
print(
    f"  Match   : {counts.get('source', 0)} == {counts.get('good', 0)} + {counts.get('quarantined', 0)} -> {counts.get('source', 0) == counts.get('good', 0) + counts.get('quarantined', 0)}"
)
print("\nThe contract IS the config. No dlt script. No manual DataFrame wrangling.")

In [ ]:
print("preview live github data - GOOD DATA")
good.limit(3)

In [ ]:
print("preview live github data - BAD DATA")
bad.limit(3)

## What You Just Saw

- **dbt adapter** — import `schema.yml` as a LakeLogic contract, zero duplication
- **dlt adapter** — declare the API in the contract, `run_source()` does extraction + validation
- **Same reconciliation guarantee** — every row accounted for regardless of source

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.